# 008 · Indefinite Integrals

| | |
|---|---|
| **Track** | Python for Calculus |
| **Level** | 3 — Integrals |
| **Prerequisites** | 007 · Derivatives |
| **Next** | 009 · Definite Integrals |

**Place in the path.** An indefinite integral is an antiderivative: a function whose derivative is the integrand.
The check is always the same. Differentiate your answer and demand the integrand back.
SymPy omits the constant \(+C\). You have to remember that the answer is a family, not a single curve.

## Learning objectives

By the end of this lesson you can:

- compute indefinite integrals of polynomials, sine, and exponentials
- use linearity to integrate a sum
- integrate by parts and by substitution, and verify each result by differentiation
- explain what SymPy leaves out when it drops \(+C\)


## 1. The definition, and the missing constant

**See.** \(\int 3x^2 \, dx = x^3 + C\), because the derivative of \(x^3\) is \(3x^2\),
and the derivative of a constant is 0. Every \(C\) works.

**Name.** `sp.integrate(expression, x)` returns one antiderivative. It does not write \(+C\).
Two antiderivatives of the same function differ by a constant. If SymPy returns \(F\) and you found \(G\),
the right test is `sp.simplify(sp.diff(F, x) - integrand) == 0`, not `F == G`.


In [ ]:
import sympy as sp

x = sp.symbols("x")
integrand = 3 * x ** 2
antiderivative = sp.integrate(integrand, x)
print("SymPy returns", antiderivative, "and leaves + C to you")
assert sp.simplify(sp.diff(antiderivative, x) - integrand) == 0
assert sp.simplify(sp.diff(antiderivative + 17, x) - integrand) == 0
print("both x**3 and x**3 + 17 differentiate back to 3x**2")


**Misconception.** SymPy's answer is the only antiderivative.

**Correction.** It is one member of the family. Adding any constant, then differentiating, is the second check.


## 2. Sine, a linear combination, and an exponential

**Do.** Verify each of these by differentiation.

\[
\int \sin x \, dx = -\cos x + C
\]

\[
\int (2x + 3) \, dx = x^2 + 3x + C
\]

\[
\int e^{x} \, dx = e^{x} + C
\]

Linearity is why the middle one splits: \(\int (2x + 3) = 2\int x + \int 3\).


In [ ]:
import sympy as sp

x = sp.symbols("x")
problems = [
    (sp.sin(x), -sp.cos(x)),
    (2 * x + 3, x ** 2 + 3 * x),
    (sp.exp(x), sp.exp(x)),
]
for integrand, by_hand in problems:
    found = sp.integrate(integrand, x)
    print(integrand, "->", found)
    assert sp.simplify(found - by_hand) == 0
    assert sp.simplify(sp.diff(found, x) - integrand) == 0
print("three antiderivatives check out")


## 3. Integration by parts

**See.** Parts is the product rule read backwards.
From \((uv)' = u'v + uv'\) you get

\[
\int u \, dv = uv - \int v \, du
\]

**Do.** \(\int x e^{x} \, dx\). Choose \(u = x\), so \(du = dx\), and \(dv = e^{x} dx\), so \(v = e^{x}\).

\[
\int x e^{x} \, dx = x e^{x} - \int e^{x} \, dx = e^{x}(x - 1) + C
\]

Lesson 007 already differentiated \(x e^{x}\) and got \(e^{x}(x+1)\), which is a different function.
Do not confuse the product rule's output with the parts output. Differentiate \(e^{x}(x-1)\) and watch \(x e^{x}\) come back.


In [ ]:
import sympy as sp

x = sp.symbols("x")
integrand = x * sp.exp(x)
found = sp.integrate(integrand, x)
by_hand = sp.exp(x) * (x - 1)
print(found)
assert sp.simplify(found - by_hand) == 0
assert sp.simplify(sp.diff(found, x) - integrand) == 0
print("parts result differentiates back to x*exp(x)")


## 4. Substitution

**See.** \(\int 2x \cos(x^2) \, dx\). The factor \(2x\) is the derivative of the inside \(x^2\).
Let \(u = x^2\). Then \(du = 2x \, dx\), and the integral becomes \(\int \cos u \, du = \sin u + C = \sin(x^2) + C\).

**Check a second way.** Differentiate \(\sin(x^2)\) with the chain rule.


In [ ]:
import sympy as sp

x = sp.symbols("x")
integrand = 2 * x * sp.cos(x ** 2)
found = sp.integrate(integrand, x)
print(found)
assert sp.simplify(found - sp.sin(x ** 2)) == 0
assert sp.simplify(sp.diff(sp.sin(x ** 2), x) - integrand) == 0
print("substitution result checks out")


### Own it

\(\int \cos x \, dx\) should come back as \(\sin x\), up to a constant.
Differentiate the answer in the same cell.


In [ ]:
import sympy as sp

x = sp.symbols("x")
found = sp.integrate(sp.cos(x), x)
print(found)
assert sp.simplify(sp.diff(found, x) - sp.cos(x)) == 0
print("checkpoint passed")


## Workshop — a family, not a formula

SymPy says \(\int 3x^2 dx = x^3\).
Show that \(x^3\), \(x^3 + 1\), and \(x^3 - 4\) are all antiderivatives,
and that the difference between any two of them is constant.
Then plot all three on \([-2, 2]\) so the "family" is visible: same shape, different heights.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import sympy as sp

x = sp.symbols("x")
integrand = 3 * x ** 2
base = sp.integrate(integrand, x)
family = [base, base + 1, base - 4]
for member in family:
    assert sp.simplify(sp.diff(member, x) - integrand) == 0
assert sp.simplify((family[1] - family[0])) == 1
assert sp.simplify((family[0] - family[2])) == 4

xs = np.linspace(-2, 2, 300)
fig, ax = plt.subplots(figsize=(7, 4))
for shift, label in ((0, r"$x^3$"), (1, r"$x^3+1$"), (-4, r"$x^3-4$")):
    ax.plot(xs, xs ** 3 + shift, linewidth=2, label=label)
ax.set_title(r"Three antiderivatives of $3x^2$. Same derivative, different constants.")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()
print("workshop passed")


## Mistakes this lesson is meant to prevent

- Dropping \(+C\) in a context that asks for the general antiderivative. SymPy will not remind you.
- Checking an integral by comparing strings. Differentiate instead.
- Choosing \(u = e^{x}\) in \(\int x e^{x} dx\), which makes parts cycle. Choose \(u\) so that \(du\) gets simpler; here \(u = x\).
- Forgetting the chain rule when differentiating \(\sin(x^2)\). The derivative is \(2x\cos(x^2)\), not \(\cos(x^2)\).

## What you now have, and what comes next

You can recover a function from its derivative and audit the recovery.
Lesson 009 puts bounds on the integral. The constant \(C\) cancels in \(F(b) - F(a)\),
which is why a definite integral is a number.
